# Introduction to yabadaba: Values

This Notebook gives an overview on Value objects and the default Value classes that are defined in yabadaba.

In [3]:
# Standard Python libraries
import datetime

from IPython.display import Markdown, display

import pandas as pd

# yabadaba imports
import yabadaba
from yabadaba.record import Record
from yabadaba import recordmanager

# yabadaba-based package in the doc folder
import yabadaba_demo

# Show yabadaba version
print('yabadaba version =', yabadaba.__version__)

# Show date of Notebook execution
print('Notebook executed on', datetime.date.today())

yabadaba version = 0.4.1
Notebook executed on 2026-10-02


## 1. Specifying Values for a Record

_add_value() parameters

- __style__ (*str*) The value style.
- __name__ (*str*) The name for the parameter value.  This corresponds to the name of the associated class attribute.
- __defaultvalue__ (*any or None, optional*) The default value to use for the property.  The default value of None indicates that there is no default value.
- __valuerequired__ (*bool, optional*) Indicates if a value must be given for the property.  If True, then checks will be performed that a value is assigned to the property.
- __allowedvalues__ (*tuple or None, optional*) A list/tuple of values that the parameter is restricted to have. Setting this to None (default) indicates any value is allowed.
- __allowcustomvalue__ (*bool, optional*) Determines how allowedvalues is interpreted. If False (default) then values are restricted to what is listed in allowedvalues.  If True, then the value is not restricted and allowedvalues becomes a list of recommended values.
- __metadatakey__ (*str, bool or None, optional*) The key name to use for the property when constructing the record metadata dict.  If set to None (default) then name will be used for metadatakey.  If set to False then the parameter will not be included in the metadata dict.
- __metadataparent__ (*str or None, optional*) If given, then this indicates that the metadatakey is actually an element of a dict in metadata with this name.  This allows for limited support for metadata having embedded dicts.
- __modelpath__ (*str, optional*) The period-delimited path after the record root element for where the parameter will be found in the built data model.  If set to None (default) then name will be used for modelpath.
- __description__ (*str or None, optional*) A short description for the value.  If not given, then the record name will be used.
- __\*\*kwargs__ (*any, optional*) Any additional style-specific keyword parameters.

In [4]:
class Example(Record):
    """Class demonstrating various _add_value() calls."""
    ########################## Basic metadata fields ##########################

    @property
    def style(self) -> str:
        """str: The record style"""
        return 'example'

    @property
    def modelroot(self):
        """str: The root element of the content"""
        return 'example'

    ############################# Define Values  ##############################

    def _init_values(self):

        self._add_value('int', 'setting',
                        description = 'a required setting that defaults to 1 and which must have a value of 1, 2, or 3',
                        defaultvalue = 1,
                        valuerequired = True,
                        allowedvalues = [1, 2, 3])
        
        self._add_value('str', 'category',
                        description = 'an optional category field with recommended values "a", "b" or "c"',
                        allowedvalues = ['a', 'b', 'c'],
                        allowcustomvalue = True)

        self._add_value('longstr', 'notes',
                        description = 'maps to notes, metadata()["notes_text"], and model["example"]["notes"]["text"]',
                        metadatakey = 'notes_text',
                        modelpath = 'notes.text')

## 2. Default Value styles

There are the different Value classes defined in yabadaba for managing different types of data.  These offer conversions and value checks, as well as having default queries associated with them. The different classes can all be loaded by specifying their style names.

- __base__ is the base Value style which does not check or convert the values being set and has no default queries.
- __str__ is for strings. A non-string value set to it will be converted to a string. The default query will only return records that exactly match a given search value. Ideal for short str options.
- __longstr__ is the same as str except that the default query returns records containing a specified substring. Better for larger, more complex text fields.
- __strlist__ is a list of strings. All values will be converted to strings if needed. The default query will limit based on if any of the values in the list match the specified search value.
- __bool__ is a boolean value. It can recognize string inputs 't', 'f', 'true', and 'false' of any upper/lower case. The default query is to return only the entries that have the specified boolean value.
- __int__ is for an integer value. It will attempt to convert non-int values to an int when set. The default query allows for exact match searches, or ranged searches by specifying a (min, max) tuple.
- __float__ is for a floating point value. It will attempt to convert non-float values to a float when set. The default query allows for exact match searches, or ranged searches by specifying a (min, max) tuple. It also has an additional "unit" parameter that specifies the unit the value is in for the metadata and model representations.
- __intarray__ is for an array of integers. It will attempt to convert values set to it to a numpy array of dtype int. No default queries. It has an additional "shape" parameter that when specifies requires that the array have a specific shape.
- __floatarray__ is for a numpy array of floating point numbers. It will attempt to convert values set to it to a numpy array of dtype float. No default queries. It has additional parameters "unit" and "shape", where "unit" specifies the unit the value is in for the metadata and model representations, and "shape" that when specified requires that the array have a specific shape.
- __date__ is for date fields. The value is represented by a datetime.date() object and supports any value that can be interpreted by it. Values in the model representation will be in the 'YEAR-MO-DY' string format. The default query searches for exact date matches in that format.
- __month__ is for representing a month of the year. The value itself is an int between 1 and 12, but recognizes strings of the English full names and 3 letter abbreviations. The model representation is consistent with the xsd:gMonth representation of '--##'. 
- __recordsubset__ is for a value that is a different yabadaba Record object. This allows for common subsets of values to be collected together and shared between multiple parent Record classes. The subset Record's queries are automatically inherited by the parent Record class.
- __recordlist__ is for a list containing Record objects all of the same class. This allows for a parent record to contain repeating subsets of values. When a recordlist value is defined, add_\<valuename\>() and \<valuename\>_df() methods are automatically added to the Record class allowing for the subsets to be set and viewed. Any queries of the values in the subset Record class are inherited by the parent Record class.
- __record__ is an alias for recordlist which is retained temporarily for backward compatibility. 


## 3. Creating new Value classes

The design of yabadaba is entirely modular making it easy to define new Value subclasses. 

### 3.1. Primary components to override

These are the methods and attributes that are most likely to be overridden when defining a new Value subclass.

- __style__ is the string style name for the Value type that yabadaba will use. *This must be set for every subclass!*
- __set_value_mod()__ specifies any value checks or data type conversions to perform when the value is being set. It is recommended that this method pass the value through __set_value_mod_textfield()__ to check if the value is coming from a XML model representation which may contain XML attributes.
- __build_model_value()__ specifies how to convert the value from its Python representation to its model representation when build_model() is called on the parent record. The model representation must consist entirely of simple data types that are JSON-compatible.
- __load_model_value()__ specifies how to convert the value from its model representation to the Python representation when load_model() is called on the parent record. Note that set_value_mod() will also operate on the value so this only needs defined if set_value_mod cannot interpret the model representation alone.
- __metadata_value()__ specifies how to convert the value from its Python representation to its metadata representation when metadata() is called on the parent record. While the metadata value can be complex, simple values are recommended for easier parsing.
- __\_default_queries__ defines a dict containing the default queries associated with the record. The dict keys are the query names and the values are Query objects. Typically, you want the Query names to be or contain the Value's name.

### 3.2. Less common methods to override

These are additional methods that subclasses can override for additional functionality, but more care is needed as the base methods perform complex operations.

- __\_\_init\_\_()__ defines how to initialize the Value. The main reason for overriding this is to add additional keyword parameters for the Value style.
- __valuedoc()__ specifies what to generate for the value when the parent Record's valuedoc is called. Change this if the default valuedoc representation based on style, name and description is lacking.
- __build_model()__ adds the value to the model representation as it is being built when the parent Record's build_model() is called. Overriding this is only necessary if the model representation requires more complex operations than what build_model_value() can do.
- __load_model()__ specifies how to read in the value from model content when the parent Record's build_model() is called. Overriding this is only necessary if the model representation requires more complex interpretation than what load_model_value() can do.
- __metadata()__ adds the value to the metadata representation as it is being built when the parent Record's metadata() is called. Overriding this is only necessary if the metadata representation requires more complex operations than what metadata_value() can do.

### 3.3. Components to not override

These are the components of the base Value class for which overriding is not recommended. These mostly consist of attributes that are set when Value objects are initialized, i.e. they are object values rather than class values.

- __name__ is the name to use for the object's Python representation, i.e. what the name of the Record attribute that points to the value is called.   
- __value__ is the Value's value which Record.name points to.
- __record__ is the Record that the Value object is part of if the Value. Automatically set when initialized using Record._add_value().  
- __defaultvalue__ is the value to assign to "value" by default. Initialization parameter.
- __valuerequired__ indicates if "value" is required (i.e. cannot be None). Initialization parameter.
- __allowedvalues__ the list of allowed values. Initialization parameter.
- __allowcustomvalue__ indicates if the value must be in allowedvalues or if allowed values is simply recommendations. Initialization parameter.
- __metadatakey__ the key to use for the Value's metadata representation if it differs from name. Initialization parameter.
- __metadataparent__ the name of the parent metadata key that the value is within if the parent is a dict or list of dict. Initialization parameter.
- __modelpath__ the path to the value (relative to the record's modelroot) for where the value is found in the model representation if it differs from name. Initialization parameter.
- __description__ is a string description of the value. Initialization parameter.
- __queries__ is a dict of queries associated with the value, where key is the query name (a.k.a. keyword parameter) and value is a Query object.  
- __add_query()__ creates and adds a new Query object to "queries". Takes the Query style, name and init parameters.
- __set_value_mod_textfield()__ should be called inside set_value_mod(). When model content is loaded, it checks the value stored at modelpath to see if it is the expected value or is a dict with a '#text' element indicating that it came from XML with XML attributes. 



### 3.4. Example: Defining a new Value for time_deltas

Below, we define a new 'time_delta' Value style for representing non-absolute periods of time.

- The TimeDeltaValue class is defined as a subclass of Value.
- The style attribute returns 'time_delta'.
- set_value_mod() first calls set_value_mod_textfield() and checks if the loaded model value is a dict with a '#text' key. Then, it checks the value's type and converts it to a datetime.timedelta object if needed.
- build_model_value() returns the string representation of the datetime.timedelta object.
- The Value class is added to yabadaba's valuemanager.loaded_styles dict. This can be done directly or by calling valuemanager's import_style() method to attemt to import from a package.

In [ ]:
import datetime

from yabadaba.value import Value
import pandas as pd

class TimeDeltaValue(Value):
    """Value object for time segments"""

    @property
    def style(self) -> str:
        """str: The value style"""
        return 'time_delta'
    
    def set_value_mod(self, val):
        
        # Check if value is in #text
        val = self.set_value_mod_textfield(val)

        # Convert to a timedelta object if needed
        if val is None:
            return None
        elif not isinstance(val, datetime.timedelta):
            val = pd.to_timedelta(val)
        return val

    def build_model_value(self):
        # Make model representation the str representation
        return str(self.value)


# Integrate the new value into yabadaba
yabadaba.valuemanager.loaded_styles['time_delta'] = TimeDeltaValue